# Week 1: What is AI? First API Call

**Lab companion to [week_01.md](../agenda/week_01.md)**

In this lab, you will:
1. Set up your OpenAI API key
2. Make your first API call
3. Understand tokens and costs
4. Experiment with different parameters

## 1. Setup

First, install the required packages:

In [ ]:
# Run this once to install dependencies
!pip install openai python-dotenv

In [ ]:
# Import libraries
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load API key from .env file
load_dotenv()

# Create client
client = OpenAI()

print("✓ Setup complete!")

## 2. Your First API Call

Let's talk to GPT! The simplest possible call:

In [ ]:
# Make your first API call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Say hello!"}
    ]
)

# Print the response
print(response.choices[0].message.content)

## 3. Understanding the Response Object

The API returns a lot of useful information. Let's explore it:

In [ ]:
# Let's look at the full response structure
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What is Python in one sentence?"}
    ]
)

print("Model used:", response.model)
print("Response ID:", response.id)
print("\n--- Token Usage ---")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print("\n--- The Answer ---")
print(response.choices[0].message.content)

## 4. Message Roles

There are three roles in a conversation:
- `system`: Sets the AI's behavior
- `user`: What you say
- `assistant`: What the AI says

In [ ]:
# Using a system message to change behavior
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a pirate. Respond in pirate speak."},
        {"role": "user", "content": "How do I learn Python?"}
    ]
)

print(response.choices[0].message.content)

## 5. Understanding Tokens

Tokens are how the AI "sees" text. Let's visualize this:

In [ ]:
# Install tiktoken for tokenization
!pip install tiktoken -q

In [ ]:
import tiktoken

# Get the tokenizer for GPT-4
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Example texts
texts = [
    "Hello",
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "Supercalifragilisticexpialidocious"
]

print("Text -> Token Count -> Tokens")
print("-" * 60)
for text in texts:
    tokens = encoding.encode(text)
    print(f"{text}")
    print(f"  → {len(tokens)} tokens: {tokens}")
    print()

## 6. Temperature: Controlling Creativity

Temperature controls randomness:
- `0.0` = Deterministic (same answer every time)
- `1.0` = More creative/varied

In [ ]:
# Low temperature (deterministic)
prompt = "Give me a creative name for a coffee shop."

print("Temperature 0 (deterministic):")
for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    print(f"  {i+1}. {response.choices[0].message.content}")

print("\nTemperature 1 (creative):")
for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1
    )
    print(f"  {i+1}. {response.choices[0].message.content}")

## 7. Controlling Output Length

Use `max_tokens` to limit response length:

In [ ]:
# Short response
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain machine learning."}],
    max_tokens=30
)
print("Max 30 tokens:")
print(response.choices[0].message.content)
print(f"(Used {response.usage.completion_tokens} tokens)")

# Longer response
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain machine learning."}],
    max_tokens=100
)
print("\nMax 100 tokens:")
print(response.choices[0].message.content)
print(f"(Used {response.usage.completion_tokens} tokens)")

## 8. Exercise: Build a Simple Helper Function

Create a reusable function to chat with the AI:

In [ ]:
def ask(question: str, system: str = "You are a helpful assistant.") -> str:
    """Simple helper to ask the AI a question."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content

# Test it!
print(ask("What is the capital of France?"))

In [ ]:
# Try different system prompts
print("As a teacher:")
print(ask("What is 2+2?", system="You are a math teacher. Explain your answer."))

print("\nAs a comedian:")
print(ask("What is 2+2?", system="You are a comedian. Make it funny."))

## 9. Calculating Costs

GPT-4o-mini pricing (as of 2024):
- Input: $0.15 per 1M tokens
- Output: $0.60 per 1M tokens

In [ ]:
def calculate_cost(response, model="gpt-4o-mini"):
    """Calculate the cost of an API call."""
    # Pricing per 1M tokens
    prices = {
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},
        "gpt-4o": {"input": 2.50, "output": 10.00},
    }
    
    p = prices.get(model, prices["gpt-4o-mini"])
    input_cost = (response.usage.prompt_tokens / 1_000_000) * p["input"]
    output_cost = (response.usage.completion_tokens / 1_000_000) * p["output"]
    
    return input_cost + output_cost

# Test
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a haiku about programming."}]
)

cost = calculate_cost(response)
print(f"Response: {response.choices[0].message.content}")
print(f"\nTokens: {response.usage.total_tokens}")
print(f"Cost: ${cost:.6f}")

## 10. Your Turn! Experiments

Try these exercises:

In [ ]:
# Exercise 1: Create a translator
# Fill in the system prompt to make the AI translate to Spanish

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "YOUR SYSTEM PROMPT HERE"},
        {"role": "user", "content": "Hello, how are you?"}
    ]
)
print(response.choices[0].message.content)

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

The system prompt tells the AI what role to play. For translation:
1. Tell it to act as a translator
2. Specify the target language (Spanish)
3. Tell it to only output the translation

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a translator. Translate to Spanish. Only output the translation."},
        {"role": "user", "content": "Hello, how are you?"}
    ]
)
print(response.choices[0].message.content)
# Output: Hola, ¿cómo estás?
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

The system prompt tells the AI what role to play. For translation:
1. Tell it to act as a translator
2. Specify the target language (Spanish)
3. Tell it to only output the translation

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a translator. Translate to Spanish. Only output the translation."},
        {"role": "user", "content": "Hello, how are you?"}
    ]
)
print(response.choices[0].message.content)
# Output: Hola, ¿cómo estás?
```

</details>

In [ ]:
# Exercise 2: Create a code explainer
# Make the AI explain code to a beginner

code = """
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Explain code simply to a beginner."},
        {"role": "user", "content": f"Explain this code:\n{code}"}
    ]
)
print(response.choices[0].message.content)

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

This one is already mostly complete! Try variations:
- Change to "Explain like I am 5"
- Add "Use analogies" to the prompt
- Ask for "step by step" explanation

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
# ELI5 version - great for beginners
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Explain code like I am 5. Use simple analogies."},
        {"role": "user", "content": f"What does this code do?\n{code}"}
    ]
)
print(response.choices[0].message.content)
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

This one is already mostly complete! Try variations:
- Change to "Explain like I am 5"
- Add "Use analogies" to the prompt
- Ask for "step by step" explanation

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
# ELI5 version - great for beginners
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Explain code like I am 5. Use simple analogies."},
        {"role": "user", "content": f"What does this code do?\n{code}"}
    ]
)
print(response.choices[0].message.content)
```

</details>

## Summary

You learned:
- ✅ How to make OpenAI API calls
- ✅ Message roles (system, user, assistant)
- ✅ How tokens work
- ✅ Temperature and max_tokens parameters
- ✅ How to calculate API costs

**Next:** [Week 2 - Build a Chatbot with Memory](week_02_chatbot.ipynb)